# 길잡이들 안전점수 계산 노트북

**계산 근거**
- 가중치: 이정현 외(2024), 「생활도로 내 보행자 사고유형별 안전 지표 분석」, 한국ITS학회논문지 23(5) 표 11의 **범죄 유형 AHP 가중치**
- 확보 가능한 4개 지표만 사용하고 가중치 합이 1이 되도록 재조정

| 지표 | 논문 가중치 | 재조정 | 방향 |
|---|---|---|---|
| 조명시설(보안등) | 0.207 | 0.414 | 많을수록 안전 |
| CCTV | 0.178 | 0.356 | 많을수록 안전 |
| 상업시설(상가) | 0.084 | 0.168 | 많을수록 안전 |
| 경사도 | 0.031 | 0.062 | 가파를수록 위험 |

**계산 흐름**
1. 분석 범위 도로망(보행) 받기 → 교차로~교차로 사이 도로 구간 단위로 계산
2. 구간 주변 시설 수 ÷ 구간 길이(100m당) = 밀도
3. 지표별 0~1 정규화 → 가중합 = 안전점수, 위험도 = 1 − 안전점수
4. 경로 비용 = 구간 길이 × (1 + 위험도)

**아직 없는 데이터가 있어도 돌아감.** clean 폴더에 없는 지표는 건너뛰고 나머지 가중치로 다시 맞춤.

## ① 설치 + 드라이브 연결

In [ ]:
!pip install -q osmnx rasterio
from google.colab import drive
drive.mount('/content/drive')

## ② 설정
반경·범위 같은 값은 **우리가 정한 가정**이라 바꿔보면서 결과 비교해도 됨 (바꾸면 기록 폴더에 남기기)

In [ ]:
BASE = "/content/drive/MyDrive/길잡이들"

# 분석 범위: 아래 장소들 중심으로 반경 RADIUS_M 안
CENTERS = {                       # 이름: (위도, 경도) — None이면 이름으로 자동 검색
    "경성대학교": None,
    "부경대학교 대연캠퍼스": None,
    "경성대·부경대역": None,
    "대연역": None,
}
RADIUS_M = 1200

# 지표 설정: 파일, 논문 가중치, 시설 탐색 반경(m), 방향
INDICATORS = {
    "light": {"file": "light_namgu.csv", "w": 0.207, "radius": 25, "higher_is_safer": True},
    "cctv":  {"file": "cctv_namgu.csv",  "w": 0.178, "radius": 50, "higher_is_safer": True},
    "shop":  {"file": "shop_namgu.csv",  "w": 0.084, "radius": 50, "higher_is_safer": True},
    "slope": {"file": "dem_namgu.tif",   "w": 0.031, "radius": None, "higher_is_safer": False},
}

MIN_LEN = 20      # 너무 짧은 구간은 밀도가 튀어서 최소 길이 20m로 계산
CLIP_Q  = 0.95    # 정규화할 때 상위 5% 극단값은 잘라서 계산
CRS_M   = "EPSG:5179"   # 거리 계산용 좌표계 (미터 단위)

## ③ 분석 범위 + 보행 도로망 받기

In [ ]:
import osmnx as ox, geopandas as gpd, pandas as pd, numpy as np, networkx as nx
from shapely.geometry import Point

pts = []
for name, xy in CENTERS.items():
    if xy is None:
        try:
            xy = ox.geocode(name + ", 부산")
        except Exception:
            raise ValueError(f"'{name}' 위치를 못 찾음 → CENTERS에 (위도, 경도) 직접 넣어줘")
    print(f"{name}: {xy}")
    pts.append(Point(xy[1], xy[0]))

area = (gpd.GeoSeries(pts, crs="EPSG:4326").to_crs(CRS_M)
          .buffer(RADIUS_M).union_all())
area_gdf = gpd.GeoDataFrame(geometry=[area], crs=CRS_M)
polygon = area_gdf.to_crs("EPSG:4326").geometry.iloc[0]

G  = ox.graph_from_polygon(polygon, network_type="walk")
Gp = ox.project_graph(G, to_crs=CRS_M)
nodes, edges = ox.graph_to_gdfs(Gp)
print("도로 구간 수:", len(edges), "/ 교차점 수:", len(nodes))

## ④ 시설 지표 계산 (보안등·CCTV·상가)
구간 주변 반경 안 시설 개수(count 합) ÷ 구간 길이 × 100 = 100m당 밀도

In [ ]:
import os

def facility_density(key, cfg):
    path = f"{BASE}/clean/{cfg['file']}"
    if not os.path.exists(path):
        print(f"[{key}] 파일 없음 → 건너뜀 ({cfg['file']})")
        return None
    df = pd.read_csv(path)
    p = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326").to_crs(CRS_M)
    buf = gpd.GeoDataFrame(geometry=edges.geometry.buffer(cfg["radius"]), crs=CRS_M)
    hit = gpd.sjoin(p[["count", "geometry"]], buf, predicate="within")
    total = hit.groupby(["u", "v", "key"])["count"].sum()
    total = total.reindex(edges.index, fill_value=0)
    length = edges["length"].clip(lower=MIN_LEN)
    print(f"[{key}] 시설 {len(p)}개 사용, 시설 있는 구간 비율 {(total > 0).mean():.0%}")
    return total / length * 100

raw_ind = {}
for key, cfg in INDICATORS.items():
    if key == "slope":
        continue
    v = facility_density(key, cfg)
    if v is not None:
        raw_ind[key] = v

## ⑤ 경사도 계산 (DEM)
DEM 파일(GeoTIFF)이 raw가 아니라 clean에 `dem_namgu.tif`로 있어야 함. 없으면 건너뜀

In [ ]:
dem_path = f"{BASE}/clean/{INDICATORS['slope']['file']}"
if os.path.exists(dem_path):
    import rasterio
    from pyproj import Transformer
    with rasterio.open(dem_path) as src:
        tf = Transformer.from_crs(CRS_M, src.crs, always_xy=True)
        xs, ys = tf.transform(nodes["x"].values, nodes["y"].values)
        z = np.array([v[0] for v in src.sample(zip(xs, ys))], dtype=float)
        z[z == src.nodata] = np.nan
    elev = pd.Series(z, index=nodes.index)
    u = edges.index.get_level_values("u"); v = edges.index.get_level_values("v")
    grade = np.abs(elev.loc[u].values - elev.loc[v].values) / edges["length"].clip(lower=MIN_LEN).values * 100
    raw_ind["slope"] = pd.Series(grade, index=edges.index).fillna(0)
    print("경사도(%) 요약:"); print(raw_ind["slope"].describe().round(1))
else:
    print("[slope] DEM 없음 → 건너뜀")

## ⑥ 정규화 + 안전점수

In [ ]:
def normalize(s, higher_is_safer):
    hi = s.quantile(CLIP_Q)
    lo = s.min()
    x = ((s.clip(upper=hi) - lo) / (hi - lo)) if hi > lo else s * 0
    return x if higher_is_safer else 1 - x

def safety_score(weights):
    used = {k: w for k, w in weights.items() if k in raw_ind}
    total_w = sum(used.values())
    score = sum(normalize(raw_ind[k], INDICATORS[k]["higher_is_safer"]) * (w / total_w) for k, w in used.items())
    return score, {k: round(w / total_w, 3) for k, w in used.items()}

paper_w = {k: c["w"] for k, c in INDICATORS.items()}
edges["safety"], used_w = safety_score(paper_w)
edges["risk"] = 1 - edges["safety"]
edges["cost"] = edges["length"] * (1 + edges["risk"])
for k in raw_ind:
    edges[f"{k}_raw"] = raw_ind[k].round(3)

# 5등급 (분위수 기준: 범위 안에서 상대적인 순위)
edges["grade"] = pd.qcut(edges["safety"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)

for c in ["cost", "risk", "safety", "grade"]:
    nx.set_edge_attributes(Gp, edges[c].to_dict(), c)
print("실제 적용 가중치:", used_w)
print(edges["safety"].describe().round(3))

## ⑦ 민감도 분석
가중치를 바꿔도 구간 순위가 크게 안 바뀌면 결과가 가중치에 덜 민감하다는 뜻 (발표 근거)

In [ ]:
from scipy.stats import spearmanr

scenarios = {
    "논문 가중치": paper_w,
    "동일 가중치": {k: 1 for k in INDICATORS},
    "조명·CCTV만": {"light": 1, "cctv": 1},
}
rows = []
for name, w in scenarios.items():
    s, uw = safety_score(w)
    g = pd.qcut(s.rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
    rows.append({"시나리오": name, "적용 가중치": uw,
                 "순위 상관(논문 대비)": round(spearmanr(s, edges["safety"])[0], 3),
                 "등급 일치율": f"{(g == edges['grade']).mean():.0%}"})
pd.DataFrame(rows)

## ⑧ 지도로 확인
1등급(초록) = 안전, 5등급(빨강) = 위험. 큰길이 위험하게 나오는 등 이상하면 설정 다시 보기

In [ ]:
import folium
colors = {1: "#1a9850", 2: "#91cf60", 3: "#fee08b", 4: "#fc8d59", 5: "#d73027"}
show = edges.to_crs("EPSG:4326").reset_index()
m = folium.Map(location=[polygon.centroid.y, polygon.centroid.x], zoom_start=15)
for _, r in show.iterrows():
    folium.PolyLine([(y, x) for x, y in r.geometry.coords], color=colors[r["grade"]], weight=3,
                    tooltip=f"등급 {r['grade']} / 안전 {r['safety']:.2f}").add_to(m)
m

## ⑨ 경로 비교 (최단 vs 안심)
출발·도착 좌표만 바꿔서 시나리오별로 돌려보기

In [ ]:
START = CENTERS.get("경성대·부경대역") or ox.geocode("경성대·부경대역, 부산")   # (위도, 경도)
END   = (35.1305, 129.1010)   # ← 원룸가 등 도착 지점 좌표로 바꾸기

tf = gpd.GeoSeries([Point(START[1], START[0]), Point(END[1], END[0])], crs="EPSG:4326").to_crs(CRS_M)
o = ox.distance.nearest_nodes(Gp, tf.iloc[0].x, tf.iloc[0].y)
d = ox.distance.nearest_nodes(Gp, tf.iloc[1].x, tf.iloc[1].y)

def summary(route):
    r = ox.routing.route_to_gdf(Gp, route, weight="cost")
    L = r["length"].sum()
    return L, (r["risk"] * r["length"]).sum() / L, r

r_short = nx.shortest_path(Gp, o, d, weight="length")
r_safe  = nx.shortest_path(Gp, o, d, weight="cost")
(L1, k1, g1), (L2, k2, g2) = summary(r_short), summary(r_safe)
print(f"최단 경로: {L1:.0f}m, 평균 위험도 {k1:.3f}")
print(f"안심 경로: {L2:.0f}m (+{L2-L1:.0f}m, 도보 약 +{(L2-L1)/67:.1f}분), 평균 위험도 {k2:.3f} ({(k1-k2)/k1:.0%} 감소)")

m2 = folium.Map(location=[START[0], START[1]], zoom_start=16)
for g, c, n in [(g1, "gray", "최단"), (g2, "blue", "안심")]:
    for geom in g.to_crs("EPSG:4326").geometry:
        folium.PolyLine([(y, x) for x, y in geom.coords], color=c, weight=5, opacity=0.7, tooltip=n).add_to(m2)
m2

## ⑩ 결과 저장 (앱에서 쓸 파일)

In [ ]:
os.makedirs(f"{BASE}/output", exist_ok=True)
cols = ["length", "safety", "risk", "grade", "cost"] + [c for c in edges.columns if c.endswith("_raw")] + ["geometry"]
out = edges[cols].reset_index().to_crs("EPSG:4326")
out.to_file(f"{BASE}/output/edges_scored.geojson", driver="GeoJSON")
print("저장:", f"{BASE}/output/edges_scored.geojson", "/", len(out), "구간")
print("적용 가중치:", used_w, "/ 반경:", RADIUS_M, "m")   # ← 이 줄 기록 폴더에 남기기